# Tau Quickstart

**Tau** infers *when* copy number amplification events (whole-genome duplications, partial genome duplications, focal gains) occurred during tumour evolution, by analysing the density of clock-like SNVs in CN segments.

This notebook walks through:
1. Setting up routes and running the full pipeline on your own data
2. Loading a pre-computed PCAWG example (double-WGD sample) and re-running clustering and visualisation
3. Interpreting the output — timing convention, event classification, discard report

**Environment:** run inside the pixi environment.
```bash
pixi run jupyter lab
```

In [ ]:
import gzip
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from tau import (
    Genome,
    CLOCK_SIGNATURES,
    preprocess_sample,
    run_sample,
    timing,
)
from tau.clustering import cluster_times_bottomup
from tau.plotting import plot_overview

print("CLOCK_SIGNATURES:", CLOCK_SIGNATURES)  # ['SBS1', 'SBS5']

---
## 1. Loading routes

Tau uses pre-computed Sage solutions (route polytopes) stored as `.sobj` files.
There are two ways to load them:

| Function | When to use |
|---|---|
| `load_routes_for_states([(major, minor), ...])` | **Recommended.** Loads only the CN states present in your sample — much faster for typical analyses. |
| `load_routes(sol_file, matrix_h5)` | Loads everything. Use when you don't know the CN states in advance, or for population-level analyses. |

> **Note:** Both functions require Sage (and Singular) on `$PATH`. The pixi environment provides this automatically.

In [ ]:
# Fast path: load only the CN states you expect in your sample.
# We use the states produced by the synthetic demo below (Section 3).
# (The first call also imports Sage and can take ~1-2 minutes.)
cn_states = [(2, 1), (2, 2), (4, 2)]
env = timing.load_routes_for_states(cn_states)

# Alternatively — load everything (needed for large PCAWG-style runs):
# env = timing.load_routes(
#     "../tau/data/solutions/",   # directory of per-state .sobj files
#     "../tau/data/matrices_7_7.h5",
# )

---
## 2. Running Tau on your own data

### Option A — one-liner

`run_sample()` chains preprocessing → EM → timing → clustering in a single call.

In [ ]:
# Replace the paths below with your own files.
# (This cell is illustrative — skip to Section 3 to run a pre-computed example.)

VCF        = "/path/to/sample.vcf"
CNV_TSV    = "/path/to/sample.cnv.tsv"
REF_FASTA  = "/path/to/hg19.fa"
COSMIC_CSV = "/path/to/COSMIC_v3.3_SBS_GRCh37.txt"
EXPOSURES  = "/path/to/exposures.tsv"  # rows=samples, cols=SBS names
PURITY     = 0.75
SAMPLE_ID  = "MY_SAMPLE"

# genome, events = run_sample(
#     SAMPLE_ID,
#     vcf_path=VCF,
#     cnv_tsv=CNV_TSV,
#     purity=PURITY,
#     env=env,
#     ref_fasta=REF_FASTA,
#     cosmic_csv=COSMIC_CSV,
#     exposures_tsv=EXPOSURES,
#     signatures=CLOCK_SIGNATURES,   # weight by SBS1 + SBS5
# )
# events

### Option B — step by step

The same pipeline broken into individual steps for inspection at each stage.

In [ ]:
# Step 1 — preprocess: VCF + CNV → per-mutation table
# mut_df = preprocess_sample(
#     sample=SAMPLE_ID,
#     vcf_path=VCF,
#     cnv_tsv=CNV_TSV,
#     purity=PURITY,
#     ref_fasta=REF_FASTA,
#     cosmic_csv=COSMIC_CSV,
#     exposures_tsv=EXPOSURES,
#     signatures=CLOCK_SIGNATURES,  # ['SBS1', 'SBS5']
#     mode="soft",    # weight by P(clock|context) — recommended
#     # mode="hard"  # binary: keep only clock-attributed mutations
# )
# mut_df.head()

# Step 2 — create Genome
# g = Genome.create(
#     mut_df,
#     purity=PURITY,
#     # normal_cn_map={"X": 1, "Y": 1}  # for male samples
# )

# Step 3 — run EM (multiplicity estimation)
# g.calculate_multiplicities(bootstrap_B=10, random_state=42)

# Step 4 — time segments
# g.time_segments(env=env)

# Step 5 — cluster events
# event_times, seg_cluster_ids, orig_times, cluster_df = cluster_times_bottomup(g)

# --- OR replace steps 3-5 with one call ---
# cluster_df = g.run(env=env)   # stores 4-tuple in g._cluster_result
# event_times, seg_cluster_ids, orig_times, cluster_df = g._cluster_result

---
## 3. Worked example: a synthetic WGD genome (no external files)

Instead of requiring a VCF/CNV bundle, we synthesise a small genome with a single
whole-genome duplication (ground-truth WGD at t = 0.30) using `tau.simulate_demo`,
then run the full pipeline on it. This is exactly what `tau demo` / `pixi run demo`
does on the command line.

> **Timing convention reminder:**  
> `t ≈ 0` = event happened **very early** (few mutations accumulated before it)  
> `t ≈ 1` = event happened **very late** (most mutations pre-date it)

In [ ]:
from tau import simulate_demo

SAMPLE = "DEMO"

# Synthesise a small genome (single WGD at t=0.30) and run EM + timing.
mut_df, purity, truth = simulate_demo(seed=0, allowed_states=cn_states)
g = Genome.create(mut_df, purity=purity, detect_min_alt=3, detect_min_vaf=0)
g.calculate_multiplicities(bootstrap_B=10, random_state=42)
g.time_segments(env=env)

print(f"Built genome: {len(g.segments)} segments, purity={purity:.2f}")
print(f"Ground-truth WGD time: {truth['WGD']:.2f}")

### Re-run clustering

The genome object already has `timing_result` on each segment.  
We re-run clustering (which is fast — no Sage needed) to get the event summary.

In [ ]:
event_times, seg_cluster_ids, orig_times, cluster_df = cluster_times_bottomup(g)
cluster_df

**Reading the cluster table**

| Column | Meaning |
|---|---|
| `time` | Event time (0 = early, 1 = late) |
| `classification` | `WGD` (≥40% of genome), `PGD` (<40%), or `chrom_specific` |
| `gf` | Genome fraction involved (as a proportion of hg19 autosomal length) |
| `n_chroms` | Number of chromosomes contributing to this event |
| `n_segments` | Number of segments assigned to this event |


### Overview plot

In [ ]:
fig = plot_overview(
    g,
    cluster_times=event_times,
    segment_cluster_ids=seg_cluster_ids,
    original_times=orig_times,
)
plt.tight_layout()
plt.show()

**Reading the overview plot**

Each row is a chromosomal segment.  The horizontal spread shows the range of timing draws.
Coloured vertical lines mark the detected events (teal = WGD, gold = PGD).
Segments whose timing draws cluster tightly around an event line are confidently timed;
wide horizontal spreads indicate underdetermined routes (multiple free variables).

---
## 4. Discard report — what was skipped and why

In [ ]:
discard_df = g.discard_report(SAMPLE)
# Returns a DataFrame with one row per discarded segment.
# Columns include: reason, chrom, start, end, major_cn, minor_cn, n_snvs, seg_len
discard_df.head()

In [ ]:
# Breakdown by CN state for low-N discards
if not discard_df.empty:
    low_n = discard_df[discard_df["reason"] == "low_effective_N"]
    if not low_n.empty:
        print("CN states most often discarded for low effective N:")
        print(
            low_n.groupby(["major_cn", "minor_cn"])
                 .size()
                 .sort_values(ascending=False)
                 .head(10)
        )

---
## 5. Extracting timing draws

`g.times_to_df(sample)` returns a long-form DataFrame of timing draws — one row per
(segment, route, draw, time-interval).  Use this for custom analyses or comparisons.

In [ ]:
times_df = g.times_to_df(SAMPLE)
print(times_df.shape)
times_df.head()

In [ ]:
# Summarised version: median + 95% CI per (segment, route, time-interval)
times_summary = g.times_to_df(SAMPLE, summarize=True, ci=(0.025, 0.5, 0.975))
times_summary.head()

### Distribution of event times across timed segments

In [ ]:
from tau.utils import order_t, pick_best_key
import numpy as np

event_t_vals = []
for seg in g.segments:
    key = pick_best_key(seg)
    if key is None:
        continue
    draws = seg.timing_result[key].get("draws", [])
    # Use only the original (non-bootstrap) draw
    base_draws = [d for d in draws if d.get("boot_id", 0) == 0]
    if not base_draws:
        continue
    cumsums = np.array([np.cumsum(order_t(d["t"]))[:-1] for d in base_draws])
    event_t_vals.append(np.median(cumsums, axis=0))

all_times = np.concatenate(event_t_vals) if event_t_vals else np.array([])

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(all_times, bins=50, color="steelblue", edgecolor="white", linewidth=0.5)
for t in event_times:
    ax.axvline(t, color="crimson", lw=1.5, linestyle="--", label=f"t={t:.2f}")
ax.set_xlabel("Event time (0=early, 1=late)")
ax.set_ylabel("Number of timed events")
ax.legend()
ax.set_title(f"{SAMPLE}")
plt.tight_layout()
plt.show()

---
## 6. Segment summary — EM multiplicity estimates

In [ ]:
# One row per segment with π and N_counts per multiplicity
seg_df = g.to_segment_summary()
print(f"{len(seg_df)} segments with EM estimates")
seg_df.head()

In [ ]:
# Distribution of CN states (major, minor) across timed segments
seg_df.groupby(["major_cn", "minor_cn"]).size().sort_values(ascending=False).head(10)

---
## Reference

**Timing convention:**
- `t_i` = fraction of total mutations in interval *i*; intervals sum to 1  
- Event time = `cumsum(t)[:-1]` — the cumulative fraction *before* the event  
- `t ≈ 0` → event happened early (few mutations pre-date it)  
- `t ≈ 1` → event happened late (most mutations pre-date it)

**Event classification thresholds:**
- `WGD` — genome fraction ≥ 40% (`wgd_thresh=0.40`)
- `PGD` — genome fraction < 40% but ≥ 2 chromosomes
- `chrom_specific` — single chromosome only

**Key parameters for `cluster_times_bottomup`:**

| Parameter | Default | Effect |
|---|---|---|
| `half_win` | 0.06 | Half-width of Poisson test window |
| `merge_tol` | 0.15 | Max gap to merge per-chrom candidates into one event |
| `match_tol` | 0.15 | Max distance to assign a segment to an event |
| `wgd_thresh` | 0.40 | Genome fraction threshold separating WGD from PGD |
| `min_ess` | 10 | Minimum EM effective N to include a segment |
